In [1]:
import os
import re
import pdfplumber
import pandas as pd
import win32com.client

# ==========================================
# Outlook Connection
# ==========================================

outlook = win32com.client.Dispatch("Outlook.Application")
namespace = outlook.GetNamespace("MAPI")

rcti_folder = namespace.Folders["iramanagendra@mmm.com"].Folders["RCTI Woolworth"]

# ==========================================
# Temporary folder for PDF attachments
# ==========================================

temp_folder = r"C:\Temp\RCTI_PDFs"
os.makedirs(temp_folder, exist_ok=True)

rows = []

# ==========================================
# Read Emails
# ==========================================

for mail in rcti_folder.Items:

    print(f"Processing Email: {mail.Subject}")

    for attachment in mail.Attachments:

        if attachment.FileName.lower().endswith(".pdf"):

            try:

                pdf_file = os.path.join(
                    temp_folder,
                    attachment.FileName
                )

                # Save attachment
                attachment.SaveAsFile(pdf_file)

                print(f"Processing PDF: {attachment.FileName}")

                invoice_details = {}

                with pdfplumber.open(pdf_file) as pdf:

                    # ==========================================
                    # Read Full PDF Text
                    # ==========================================

                    full_text = ""

                    for page in pdf.pages:
                        text = page.extract_text()

                        if text:
                            full_text += text + "\n"

                    # ==========================================
                    # Extract Invoice Header
                    # ==========================================

                    invoice_match = re.search(
                        r'Our Reference:\s*(.*?)\s*'
                        r'Invoice No:\s*(.*?)\s*'
                        r'Invoice Date:\s*(.*?)\s*'
                        r'Invoicing Party:\s*(.*?)\s*'
                        r'Site No:\s*(.*?)\s*'
                        r'Site Name:\s*(.*?)\s*Recipient Created Tax Invoice',
                        full_text,
                        re.DOTALL
                    )

                    if invoice_match:

                        invoice_details = {
                            "Our_Reference": invoice_match.group(1).strip(),
                            "Invoice_No": invoice_match.group(2).strip(),
                            "Invoice_Date": invoice_match.group(3).strip(),
                            "Invoicing_Party": invoice_match.group(4).strip(),
                            "Site_No": invoice_match.group(5).strip(),
                            "Site_Name": invoice_match.group(6).strip()
                        }

                    # ==========================================
                    # Line Item Pattern
                    # ==========================================

                    pattern = re.compile(
                        r'^(.*?)\s+'
                        r'(\d+\.\d{3})\s+'
                        r'([A-Z0-9]+)\s+'
                        r'(\d+\.\d{3})\s+'
                        r'(\d+\.\d{2})\s+'
                        r'(\d+\.\d{3})\s+'
                        r'([\d,]+\.\d{2})\s+'
                        r'([\d,]+\.\d{2})\s+'
                        r'([\d,]+\.\d{2})$'
                    )

                    # ==========================================
                    # Process Line Items
                    # ==========================================

                    for page in pdf.pages:

                        text = page.extract_text()

                        if not text:
                            continue

                        for line in text.split("\n"):

                            line = line.strip()

                            if any(skip_text in line for skip_text in [
                                "Recipient Created Tax Invoice",
                                "Balance carry forward",
                                "Total GST is payable",
                                "Summary of Recipient",
                                "SUPPLIER ADDRESS"
                            ]):
                                continue

                            match = pattern.search(line)

                            if match:

                                rows.append({
                                    **invoice_details,
                                    "Email_Subject": mail.Subject,
                                    "PDF_File": attachment.FileName,
                                    "Item_Details": match.group(1),
                                    "PO_Qty": match.group(2),
                                    "PO_UOM": match.group(3),
                                    "Cost_Qty": match.group(4),
                                    "GR_Price": match.group(5),
                                    "GR_Rec": match.group(6),
                                    "Merch_Amt": match.group(7),
                                    "GST_Amt": match.group(8),
                                    "Total_Amt": match.group(9)
                                })

                # Delete temp PDF after processing
                os.remove(pdf_file)

            except Exception as e:
                print(f"Error processing {attachment.FileName}: {e}")

# ==========================================
# Create DataFrame
# ==========================================

df = pd.DataFrame(rows)

# ==========================================
# Convert Numeric Columns
# ==========================================

numeric_cols = [
    "PO_Qty",
    "Cost_Qty",
    "GR_Price",
    "GR_Rec",
    "Merch_Amt",
    "GST_Amt",
    "Total_Amt"
]

for col in numeric_cols:

    if col in df.columns:

        df[col] = (
            df[col]
            .astype(str)
            .str.replace(",", "", regex=False)
            .str.strip()
        )

        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

# ==========================================
# Export Excel
# ==========================================

output_file = r"C:\Users\AD1HCZZ\3M\ANZ Invoice Reconciliation - ANZ RCTI Email Data\RCTI_Data2.xlsx"

df.to_excel(
    output_file,
    index=False
)

print("\n===================================")
print("Rows Extracted:", len(df))
print("Output File:", output_file)
print("===================================")

print(df.head())

Processing Email: Recipient Created Tax Invoice and Adjustment Notes
Processing PDF: 51975598742027.pdf
Processing Email: [EXTERNAL] Recipient Created Tax Invoice and Adjustment Notes
Processing PDF: 51978886042027.pdf
Processing Email: [EXTERNAL] Recipient Created Tax Invoice and Adjustment Notes
Processing PDF: 51974941772027.pdf
Processing Email: [EXTERNAL] sRCTI PDF
Processing PDF: 6136_21260827_sRCTI.pdf

Rows Extracted: 56
Output File: C:\Users\AD1HCZZ\3M\ANZ Invoice Reconciliation - ANZ RCTI Email Data\RCTI_Data2.xlsx
                        Our_Reference                       Invoice_No  \
0  31998670\nBuilding A 1 Rivett Road  5197559874\nNORTH RYDE NSW 2113   
1  31998670\nBuilding A 1 Rivett Road  5197559874\nNORTH RYDE NSW 2113   
2  31998670\nBuilding A 1 Rivett Road  5197559874\nNORTH RYDE NSW 2113   
3  31998670\nBuilding A 1 Rivett Road  5197559874\nNORTH RYDE NSW 2113   
4  31998670\nBuilding A 1 Rivett Road  5197559874\nNORTH RYDE NSW 2113   

            Invoice_Date